# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

%pip install docling markdown-it-py
%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Document
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
with open(glob.glob(f"{data_dir}/*.md")[0], "r") as f:
    text = f.read()

In [ ]:
# Step 4: Text Chunking and Dataset Creation

from pathlib import Path
from typing import List
import os
import re

import datasets
from markdown_it import MarkdownIt

from sdg_hub import Flow


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    # This ensures we don't split in the middle of headings, lists, etc.
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # Emit a complete chunk
                chunks.append(" ".join(current_words))
                # Prepare next buffer with overlap from the end of this chunk
                # This ensures context continuity between chunks
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def set_model_config(flow_object: Flow) -> Flow:
    """
    Configure model settings for SDG Hub flows using environment variables.
    """
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")

    if model_provider == "hosted_vllm":
        flow_object.set_model_config(
            model=os.getenv("VLLM_MODEL", "openai/RedHatAI/gpt-oss-20b"),
            api_base=os.getenv("VLLM_API_BASE", "http://inference-server-ai-inference-server-ocp.apps.cluster-s972k.s972k.sandbox1774.opentlc.com/v1"),
            api_key=os.getenv("VLLM_API_KEY", "EMPTY"),
            enable_reasoning=os.getenv("ENABLE_REASONING", "false").lower() in ("1", "true", "yes"),
        )
    elif model_provider == "openai":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    elif model_provider == "openrouter":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
            api_base="https://openrouter.ai/api/v1",
        )
    elif model_provider == "ollama":
        flow_object.set_model_config(
            model=os.getenv("OLLAMA_MODEL", "ollama/gemma2"),
            api_base=os.getenv("OLLAMA_API_BASE", "http://localhost:11434"),
        )
    elif model_provider == "maas":
        flow_object.set_model_config(
            model=os.getenv("MAAS_MODEL"),
            api_base=os.getenv("MAAS_API_BASE"),
            api_key=os.getenv("MAAS_API_KEY"),
        )
    else:
        raise ValueError(f"Unsupported MODEL_PROVIDER: {model_provider}")

    return flow_object


DOMAIN_LABELS = {
    "finance",
    "healthcare",
    "legal",
    "technology",
    "education",
    "government",
    "science",
    "operations",
    "hr",
    "general",
}

DOMAIN_ALIAS_MAP = {
    "human resources": "hr",
    "rh": "hr",
    "people": "hr",
    "public sector": "government",
    "public policy": "government",
    "compliance": "legal",
}


def get_icl_flow_path(flow_file_name: str) -> Path:
    candidates = [
        Path(f"icl_generation/{flow_file_name}"),
        Path(f"examples/knowledge_tuning/enhanced_summary_knowledge_tuning/icl_generation/{flow_file_name}"),
    ]
    flow_path = next((path for path in candidates if path.exists()), None)
    if flow_path is None:
        raise FileNotFoundError(f"ICL flow file not found: {flow_file_name}")
    return flow_path


def run_icl_flow(
    flow_file_name: str,
    flow_input: datasets.Dataset,
    runtime_params: dict,
):
    flow = Flow.from_yaml(str(get_icl_flow_path(flow_file_name)))
    flow = set_model_config(flow)
    return flow.generate(flow_input, runtime_params=runtime_params, max_concurrency=1)


def to_scalar(value):
    if isinstance(value, list):
        return value[0] if value else ""
    return value


def parse_icl_queries(raw_queries: str, max_queries: int) -> List[str]:
    """
    Parse question bullets from LLM output into a deduplicated list.
    """
    parsed = []
    for line in raw_queries.splitlines():
        cleaned = line.strip()
        # Remove common list markers generated by LLMs (supports nested bullets).
        while True:
            stripped = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", cleaned).strip()
            if stripped == cleaned:
                break
            cleaned = stripped
        if cleaned:
            parsed.append(cleaned)

    unique = []
    seen = set()
    for question in parsed:
        norm = question.lower()
        if norm not in seen:
            seen.add(norm)
            unique.append(question)

    return unique[:max_queries]


def format_queries_for_prompt(queries: List[str]) -> str:
    return "\n".join(f"- {query}" for query in queries)


def normalize_domain(raw_domain: str) -> str:
    normalized = (raw_domain or "").strip().lower()
    if normalized in DOMAIN_LABELS:
        return normalized
    if normalized in DOMAIN_ALIAS_MAP:
        return DOMAIN_ALIAS_MAP[normalized]

    for alias, canonical in DOMAIN_ALIAS_MAP.items():
        if alias in normalized:
            return canonical

    return "general"


def parse_icl_candidate(generated_row, max_queries: int) -> dict:
    candidate = {
        "document_outline": str(to_scalar(generated_row.get("document_outline", ""))).strip(),
        "icl_document": str(to_scalar(generated_row.get("icl_document", ""))).strip(),
        "icl_queries": str(to_scalar(generated_row.get("icl_queries", ""))).strip(),
        "domain": normalize_domain(str(to_scalar(generated_row.get("domain", "")))),
    }

    queries = parse_icl_queries(candidate.get("icl_queries", ""), max_queries)
    if not queries:
        outline = candidate.get("document_outline", "this document") or "this document"
        queries = [f"What are the key points covered in '{outline}'?"]

    candidate["icl_queries"] = queries
    candidate["icl_query_count"] = len(queries)
    candidate["icl_queries_text"] = format_queries_for_prompt(queries)
    return candidate


def extract_tag_content(text: str, start_tag: str, end_tag: str) -> str:
    pattern = re.escape(start_tag) + r"(.*?)" + re.escape(end_tag)
    match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""


def validate_icl_candidate(source_document: str, candidate: dict) -> tuple[bool, str]:
    validation_input = datasets.Dataset.from_dict(
        {
            "document": [source_document],
            "document_outline": [candidate["document_outline"]],
            "icl_document": [candidate["icl_document"]],
            "icl_queries_text": [candidate["icl_queries_text"]],
            "domain": [candidate["domain"]],
        }
    )

    try:
        validation_result = run_icl_flow(
            "validate_flow.yaml",
            validation_input,
            runtime_params={"validate_icl": {"max_tokens": 1024}},
        )
    except Exception as exc:
        return True, f"Validation skipped due flow error: {type(exc).__name__}: {exc}"

    if len(validation_result) == 0:
        return True, "Validation skipped: validator returned no rows"

    row = validation_result[0]
    raw_validation = str(to_scalar(row.get("validation_raw_text", ""))).strip()

    # Backward compatibility with old validator schema.
    if not raw_validation:
        validation_pass = str(to_scalar(row.get("validation_pass", "NO"))).strip().upper() == "YES"
        feedback = str(to_scalar(row.get("validation_feedback", ""))).strip() or "Validator returned empty feedback"
        return validation_pass, feedback

    tagged_pass = extract_tag_content(raw_validation, "[VALIDATION_PASS]", "[/VALIDATION_PASS]").upper()
    tagged_feedback = extract_tag_content(raw_validation, "[VALIDATION_FEEDBACK]", "[/VALIDATION_FEEDBACK]")

    if tagged_pass in {"YES", "NO"}:
        validation_pass = tagged_pass == "YES"
        feedback = tagged_feedback or ("OK" if validation_pass else "Validation failed without detailed feedback")
        return validation_pass, feedback

    # Fail-open when validator output is unstructured to avoid crashing preprocessing.
    return True, "Validation skipped: unstructured validator output"


def repair_icl_candidate(source_document: str, candidate: dict, feedback: str, max_queries: int) -> dict:
    repair_input = datasets.Dataset.from_dict(
        {
            "document": [source_document],
            "document_outline": [candidate["document_outline"]],
            "icl_document": [candidate["icl_document"]],
            "icl_queries_text": [candidate["icl_queries_text"]],
            "domain": [candidate["domain"]],
            "validation_feedback": [feedback],
            "icl_max_queries": [max_queries],
        }
    )

    repaired_result = run_icl_flow(
        "repair_flow.yaml",
        repair_input,
        runtime_params={"repair_icl": {"max_tokens": 1536}},
    )

    if len(repaired_result) == 0:
        raise ValueError("ICL repair flow returned no rows.")

    return parse_icl_candidate(repaired_result[0], max_queries)


def build_icl_with_sdg_hub(markdown_text: str) -> dict:
    """
    Generate ICL v2 with SDG Hub orchestration:
    generate -> validate -> repair (if needed) -> revalidate.
    """
    icl_input_document = markdown_text.strip()[:20000]
    if not icl_input_document:
        raise ValueError("Processed markdown is empty. Cannot generate ICL.")

    try:
        max_queries = max(1, int(os.getenv("ICL_MAX_QUERIES", "5")))
    except ValueError:
        max_queries = 5

    generation_input = datasets.Dataset.from_dict(
        {
            "document": [icl_input_document],
            "icl_max_queries": [max_queries],
        }
    )

    generated_icl = run_icl_flow(
        "flow.yaml",
        generation_input,
        runtime_params={"gen_icl": {"max_tokens": 1536}},
    )

    if len(generated_icl) == 0:
        raise ValueError("ICL generation flow returned no rows.")

    candidate = parse_icl_candidate(generated_icl[0], max_queries)
    validation_pass, validation_feedback = validate_icl_candidate(icl_input_document, candidate)
    repaired = False

    if not validation_pass:
        candidate = repair_icl_candidate(
            source_document=icl_input_document,
            candidate=candidate,
            feedback=validation_feedback,
            max_queries=max_queries,
        )
        repaired = True
        validation_pass, validation_feedback = validate_icl_candidate(icl_input_document, candidate)

    # Keep backward compatibility with existing enhanced_multi_summary_qa flows.
    for idx in range(3):
        candidate[f"icl_query_{idx + 1}"] = (
            candidate["icl_queries"][idx] if idx < len(candidate["icl_queries"]) else candidate["icl_queries"][-1]
        )

    candidate["icl_validation_pass"] = "YES" if validation_pass else "NO"
    candidate["icl_validation_feedback"] = validation_feedback
    candidate["icl_repaired"] = repaired

    if not candidate["domain"]:
        candidate["domain"] = "general"

    return candidate


chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)
if not chunks and text.strip():
    chunks = [text.strip()]


# Prepare seed data for the SDG-Hub knowledge pipeline.
#
# The seed data requires the following fields:
#   - document_outline: A concise title or summary that accurately represents the entire document.
#     For documents covering multiple themes, consider providing multiple outlines (one per section).
#   - icl_document: A representative sample extract from the document. This may include tables, code snippets, definitions, etc.
#   - icl_queries / icl_query_count: LLM-decided question list (up to ICL_MAX_QUERIES from .env).
#   - icl_query_1, icl_query_2, icl_query_3: Compatibility fields for downstream enhanced_multi_summary_qa flows.
#   - icl_validation_pass / icl_validation_feedback / icl_repaired: v2 quality control metadata.
#   - domain: The domain or subject area of the document.
#
# The code below creates a HuggingFace Dataset from the document chunks,
# then maps the required ICL fields to each entry, and finally saves the result as a JSONL file.

seed_data = datasets.Dataset.from_dict({"document": chunks})
icl = build_icl_with_sdg_hub(text)
if not icl["icl_document"] and chunks:
    icl["icl_document"] = chunks[0][:3000]

# Map the ICL fields to each document chunk (if you want to use the same ICL for all, as shown here)
seed_data = seed_data.map(lambda x: icl)

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)

### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook